## RAG Setup
The notebook is created to demonstrate the RAG setup using LlamaIndex.

In [9]:
import os
os.chdir("..")

In [10]:
from llama_index.core import SimpleDirectoryReader
documents = SimpleDirectoryReader("rag_data").load_data()

In [11]:
len(documents)
type(documents[0])

llama_index.core.schema.Document

In [12]:
## To view the metadata of the document
# documents[2].__dict__

In [13]:
from llama_index.core.node_parser import SentenceSplitter

parser = SentenceSplitter(
    chunk_size=128, # in tokens
    chunk_overlap=16, #in tokens
    paragraph_separator="\n\n"
)

nodes = parser.get_nodes_from_documents(documents, show_progress=True)

Parsing nodes:   0%|          | 0/72 [00:00<?, ?it/s]

In [14]:
type(nodes[42])

llama_index.core.schema.TextNode

In [15]:
nodes[42].__dict__

{'id_': 'ef450c56-a82a-42c9-a0a8-b532ff789895',
 'embedding': None,
 'metadata': {'page_label': '16648',
  'file_name': '2023.emnlp-main.1036.pdf',
  'file_path': '/Users/zp3077/Personal/Projects/translations_exploration/rag_data/2023.emnlp-main.1036.pdf',
  'file_type': 'application/pdf',
  'file_size': 1122216,
  'creation_date': '2025-03-31',
  'last_modified_date': '2025-01-16'},
 'excluded_embed_metadata_keys': ['file_name',
  'file_type',
  'file_size',
  'creation_date',
  'last_modified_date',
  'last_accessed_date'],
 'excluded_llm_metadata_keys': ['file_name',
  'file_type',
  'file_size',
  'creation_date',
  'last_modified_date',
  'last_accessed_date'],
 'relationships': {<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='7b93d4e5-5d7f-429a-a2bd-840fa1ea3eed', node_type=<ObjectType.DOCUMENT: '4'>, metadata={'page_label': '16648', 'file_name': '2023.emnlp-main.1036.pdf', 'file_path': '/Users/zp3077/Personal/Projects/translations_exploration/rag_data/2023.emnlp-main.10

## Embedding using the MPNET


In [16]:
!pip install llama-index-embeddings-huggingface

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)



[notice] A new release of pip is available: 23.0.1 -> 25.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import Settings
hf_embed_model = HuggingFaceEmbedding(model_name="sentence-transformers/all-mpnet-base-v2")


In [3]:
example_embedding = hf_embed_model.get_text_embedding("i am batman")

In [4]:
example_embedding

[0.04616103693842888,
 0.007041321136057377,
 0.0020304268691688776,
 -0.008090358227491379,
 -0.005201134365051985,
 -0.006765285972505808,
 -0.003343130461871624,
 -0.0027625018265098333,
 -0.004663561470806599,
 0.028447473421692848,
 0.02495879866182804,
 0.035393375903367996,
 0.02088375762104988,
 -0.031648196280002594,
 0.06325724720954895,
 -0.04133504256606102,
 0.061001185327768326,
 -0.012472913600504398,
 0.0349886417388916,
 -0.012564721517264843,
 0.023047536611557007,
 0.04985233023762703,
 -0.03602089732885361,
 0.007831253111362457,
 -2.816654887283221e-05,
 -0.01868806593120098,
 0.08226693421602249,
 -0.0043524946086108685,
 0.01140710897743702,
 0.03500700369477272,
 0.006675284821540117,
 -0.01610858365893364,
 0.030415860936045647,
 0.006658222060650587,
 1.7344352727377554e-06,
 0.00802542269229889,
 0.023620419204235077,
 0.005435038823634386,
 -0.009657633490860462,
 -0.007695379666984081,
 -0.05297747999429703,
 -0.007173546589910984,
 -0.016390856355428696,
 

In [5]:
def get_embedding_dimensions(embed_model, list_of_strings):
    embeddings = embed_model.get_text_embedding_batch(list_of_strings)
    embed_lens = []
    for embedding in embeddings:
        embed_lens.append(len(embedding))
    return embed_lens

In [6]:
get_embedding_dimensions(hf_embed_model, ["i am batman", "i am superman"])

[768, 768]

In [7]:
hf_embed_model.similarity(
    hf_embed_model.get_text_embedding("""I am jhn"""),
    hf_embed_model.get_text_embedding("i am batman"),
    mode="cosine"
)

np.float64(0.2932456871122807)

## Indexing process

In [22]:
from llama_index.core import Document

full_document = Document(text=documents[0].text)



In [27]:
from llama_index.core import Document, VectorStoreIndex
# index = VectorStoreIndex.from_documents(
#     # remember, you must pass a list of documents!
#     nodes,
#     embed_model=hf_embed_model,
#     show_progress=True)

AttributeError: 'TextNode' object has no attribute 'get_doc_id'

In [28]:
# create the index from the nodes
index_from_nodes = VectorStoreIndex(
    nodes,
    embed_model=hf_embed_model,
    show_progress=True
)

Generating embeddings:   0%|          | 0/1142 [00:00<?, ?it/s]

In [ ]:
index_from_nodes

## Store data

In [35]:
import faiss
# Define the dimensions of your embeddings
d = 768
faiss_index = faiss.IndexFlatL2(d)

In [36]:
from llama_index.vector_stores.faiss import FaissVectorStore
from llama_index.core import StorageContext

vector_store = FaissVectorStore(faiss_index=faiss_index)
storage_context = StorageContext.from_defaults(vector_store=vector_store)
index_from_nodes = VectorStoreIndex(
    nodes,
    embed_model=hf_embed_model,
    show_progress=True,
    storage_context = storage_context
)

Generating embeddings:   0%|          | 0/1142 [00:00<?, ?it/s]

In [37]:
## Also we can do like this

index_from_nodes_doc = VectorStoreIndex.from_documents(
    documents,
    embed_model=hf_embed_model,
    show_progress=True,
    storage_context = storage_context
)


Parsing nodes:   0%|          | 0/72 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/125 [00:00<?, ?it/s]

In [38]:
index_from_nodes == index_from_nodes_doc

False

## Storing the FAISS database on the local


In [50]:

index_from_nodes.storage_context.persist(persist_dir="data/index_store")


## Retrieving the FAISS database from the local

In [60]:
retriever = index_from_nodes.as_retriever( similarity_top_k=5)

In [62]:
response = retriever.retrieve("What is the bleu score?")
for node in response:
    print(node.get_text())


The potential reasons may be:
(1) d-BLEU only measures the similarity of the
n-grams between the MT output and the reference
translations. However, human takes into account
additional factors such as coherence, fluency, and
naturalness of the translation, which may not neces-
sarily correlate with d-BLEU scores.
28 - 0.60 82.88 + 1.98
Table 3:BLEU scores on En→Fr and En→De test sets with
single and multiple references.
is caused by the auxiliary information negatively
impacting the original input, either by introducing
noise or by increasing the complexity of the input
text.
Furthermore, we evaluate the perfor-
mance of our system using multiple metrics, includ-
ing BLEU (Papineni et al., 2002), ChrF++ (Popovi´c,
2017) and TER (Snover et al., 2006). In this work,
we report BLEU scores using SacreBLEU (Post,
2018).
Association for Computational
Linguistics.
Matt Post. 2018. A call for clarity in reporting BLEU
scores. In Proceedings of the Third Conference on
Machine Translation: Resear

## using the query engine
- We should an LLM from openAi, Cohere, or Mistral as these are LLm which are supported by LlamaIndex.

In [ ]:
## Using the EUroLLM as the LLM
from transformers import AutoModelForCausalLM, AutoTokenizer
# from llama_index.llms import BaseLLM
#
# # Load EuroLLM from Hugging Face (assuming it's available there)
# llm = Llama.from_pretrained(
#     repo_id="Triangle104/EuroLLM-9B-Q8_0-GGUF",
#     filename="eurollm-9b-q8_0.gguf",
# )
#
# # Create a custom LLM wrapper
# class EuroLLM(BaseLLM):
#     def complete(self, prompt: str) -> str:
#         inputs = tokenizer(prompt, return_tensors="pt")
#         outputs = model.generate(inputs["input_ids"], max_length=200)
#         response = tokenizer.decode(outputs[0], skip_special_tokens=True)
#         return response
#
# # Use EuroLLM in your query engine
# llm = EuroLLM()

In [ ]:
from llama_index.llms.

llm = Cohere(model="command-r-plus")

query_engine = index.as_query_engine(llm=llm, streaming=True)

response = query_engine.query(
    "What do the Sikh Stoics believe?"
)

response.print_response_stream()